# Feature Extraction — Titanic Dataset

This notebook continues from the previous preprocessing steps and uses:

```text
titanic_transformed-Scaled.csv
```

**Feature Extraction** means creating new features from existing information in the dataset.

In this notebook, we will create a few useful Titanic features without going deeply into advanced feature engineering.

## 1. Import Libraries

In [64]:
import pandas as pd

pd.set_option("display.max_columns", None)

## 2. Load the Dataset

Based on the project structure:

```text
03-Data-Preprocessing/
├── Dataset/
│   └── titanic_transformed-Scaled.csv
└── 05-Feature-Engineering/
    └── 02-Feature-Extraction/
        └── Feature-Extraction.ipynb
```

So the relative path is:

In [ ]:
df = pd.read_csv("../../Dataset/04_titanic_transformed-Scaled.csv")

df.head()

,Fare,Age,Pclass,SibSp,Parch,Sex_male,Embarked_Q,Embarked_S,Survived,Name,Ticket
0,-0.312011,-0.565736,1.0,0.125,0.0,1.0,0.0,1.0,0,"Braund, Mr. Owen Harris",A/5 21171
1,2.461242,0.663861,0.0,0.125,0.0,0.0,0.0,0.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",PC 17599
2,-0.282777,-0.258337,1.0,0.000,0.0,0.0,0.0,1.0,1,"Heikkinen, Miss. Laina",STON/O2. 3101282
3,1.673732,0.433312,0.0,0.125,0.0,0.0,0.0,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",113803
4,-0.277363,0.433312,1.0,0.000,0.0,1.0,0.0,1.0,0,"Allen, Mr. William Henry",373450


## 3. Inspect the Dataset

In [66]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (891, 11)

Columns:
['Fare', 'Age', 'Pclass', 'SibSp', 'Parch', 'Sex_male', 'Embarked_Q', 'Embarked_S', 'Survived', 'Name', 'Ticket']


## 4. What Is Feature Extraction?

Feature Extraction creates **new features from existing information**.

Examples from the Titanic dataset:

```text
SibSp + Parch
      ↓
FamilySize

FamilySize
      ↓
IsAlone

Name
 ↓
Title

Ticket
 ↓
TicketGroupSize
```

### Difference

```text
Feature Selection  → Choose existing features
Feature Extraction → Create new features
```

## 5. Extract `FamilySize`

`SibSp` = siblings/spouses aboard

`Parch` = parents/children aboard

We can combine them with the passenger themselves:

\[
FamilySize = SibSp + Parch + 1
\]

In [67]:
df_extracted = df.copy()

df_extracted["FamilySize"] = (
    df_extracted["SibSp"] +
    df_extracted["Parch"] +
    1
)

df_extracted[["SibSp", "Parch", "FamilySize"]].head()

,SibSp,Parch,FamilySize
0,0.125,0.0,1.125
1,0.125,0.0,1.125
2,0.000,0.0,1.000
3,0.125,0.0,1.125
4,0.000,0.0,1.000


## 6. Extract `IsAlone`

A passenger is alone when `FamilySize = 1`.

```text
FamilySize = 1 → Alone
FamilySize > 1 → Not Alone
```

In [68]:
df_extracted["IsAlone"] = (
    df_extracted["FamilySize"] == 1
).astype(int)

df_extracted[["FamilySize", "IsAlone"]].head()

,FamilySize,IsAlone
0,1.125,0
1,1.125,0
2,1.000,1
3,1.125,0
4,1.000,1


## 7. Extract `Title` from `Name`

The `Name` column contains titles such as:

```text
Mr.
Mrs.
Miss.
Master.
```

We can extract the title from each passenger's name.

### **Note : I have to read and Store titanic_encoded as df_Name because of the Name and Ticker Columns**

In [69]:
df["Title"] = (
    df["Name"]
    .str.extract(r",\s*([^.]*)\.")[0]
    .str.strip()
)

df[["Name", "Title"]].head(10)

,Name,Title
0,"Braund, Mr. Owen Harris",Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,"Heikkinen, Miss. Laina",Miss
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs
4,"Allen, Mr. William Henry",Mr
5,"Moran, Mr. James",Mr
6,"McCarthy, Mr. Timothy J",Mr
7,"Palsson, Master. Gosta Leonard",Master
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",Mrs
9,"Nasser, Mrs. Nicholas (Adele Achem)",Mrs


## 8. Inspect the Extracted Titles

In [70]:
df["Title"].value_counts().head(15)

Title
Mr        517
Miss      182
Mrs       125
Master     40
Dr          7
Rev         6
Major       2
Mlle        2
Col         2
Don         1
Mme         1
Ms          1
Lady        1
Sir         1
Capt        1
Name: count, dtype: int64

## 9. Extract `TicketGroupSize`

Passengers with the same ticket may have travelled together.

We can count how many passengers share each ticket.

In [71]:
ticket_counts = df["Ticket"].value_counts()

df["TicketGroupSize"] = (
    df["Ticket"].map(ticket_counts)
)

df[["Ticket", "TicketGroupSize"]].head(10)

,Ticket,TicketGroupSize
0,A/5 21171,1
1,PC 17599,1
2,STON/O2. 3101282,1
3,113803,2
4,373450,1
5,330877,1
6,17463,1
7,349909,4
8,347742,3
9,237736,2


In [72]:
df

,Fare,Age,Pclass,SibSp,Parch,Sex_male,Embarked_Q,Embarked_S,Survived,Name,Ticket,Title,TicketGroupSize
0,-0.312011,-0.565736,1.0,0.125,0.000000,1.0,0.0,1.0,0,"Braund, Mr. Owen Harris",A/5 21171,Mr,1
1,2.461242,0.663861,0.0,0.125,0.000000,0.0,0.0,0.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",PC 17599,Mrs,1
2,-0.282777,-0.258337,1.0,0.000,0.000000,0.0,0.0,1.0,1,"Heikkinen, Miss. Laina",STON/O2. 3101282,Miss,1
3,1.673732,0.433312,0.0,0.125,0.000000,0.0,0.0,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",113803,Mrs,2
4,-0.277363,0.433312,1.0,0.000,0.000000,1.0,0.0,1.0,0,"Allen, Mr. William Henry",373450,Mr,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,-0.062981,-0.181487,0.5,0.000,0.000000,1.0,0.0,1.0,0,"Montvila, Rev. Juozas",211536,Rev,1
887,0.673281,-0.796286,0.0,0.000,0.000000,0.0,0.0,1.0,1,"Graham, Miss. Margaret Edith",112053,Miss,1
888,0.389604,-0.104637,1.0,0.125,0.333333,0.0,0.0,1.0,0,"Johnston, Miss. Catherine Helen ""Carrie""",W./C. 6607,Miss,2
889,0.673281,-0.258337,0.0,0.000,0.000000,1.0,0.0,0.0,1,"Behr, Mr. Karl Howell",111369,Mr,1


## 10. Review the New Features

We have created:

```text
FamilySize
IsAlone
Title
TicketGroupSize
```

In [73]:
new_features = [
    "FamilySize",
    "IsAlone",
    "Title",
    "TicketGroupSize"
]
df_extracted["Title"] = df["Title"]
df_extracted["TicketGroupSize"] = df["TicketGroupSize"]

df_extracted[new_features].head(10)

,FamilySize,IsAlone,Title,TicketGroupSize
0,1.125000,0,Mr,1
1,1.125000,0,Mrs,1
2,1.000000,1,Miss,1
3,1.125000,0,Mrs,2
4,1.000000,1,Mr,1
5,1.000000,1,Mr,1
6,1.000000,1,Mr,1
7,1.541667,0,Master,4
8,1.333333,0,Mrs,3
9,1.125000,0,Mrs,2


## 11. Remove Raw Columns Used for Extraction

For this basic pipeline, we can remove `Name` and `Ticket` after extracting useful information from them.

The new structured features remain in the dataset.

In [74]:
df_extracted = df_extracted.drop(
    columns=["Name", "Ticket"]
)

df_extracted.head()

,Fare,Age,Pclass,SibSp,Parch,Sex_male,Embarked_Q,Embarked_S,Survived,FamilySize,IsAlone,Title,TicketGroupSize
0,-0.312011,-0.565736,1.0,0.125,0.0,1.0,0.0,1.0,0,1.125,0,Mr,1
1,2.461242,0.663861,0.0,0.125,0.0,0.0,0.0,0.0,1,1.125,0,Mrs,1
2,-0.282777,-0.258337,1.0,0.000,0.0,0.0,0.0,1.0,1,1.000,1,Miss,1
3,1.673732,0.433312,0.0,0.125,0.0,0.0,0.0,1.0,1,1.125,0,Mrs,2
4,-0.277363,0.433312,1.0,0.000,0.0,1.0,0.0,1.0,0,1.000,1,Mr,1


## 12. Final Verification

In [75]:
print("Shape:", df_extracted.shape)

print("\nColumns:")
print(df_extracted.columns.tolist())

print("\nMissing values:")
print(df_extracted.isnull().sum())

Shape: (891, 13)

Columns:
['Fare', 'Age', 'Pclass', 'SibSp', 'Parch', 'Sex_male', 'Embarked_Q', 'Embarked_S', 'Survived', 'FamilySize', 'IsAlone', 'Title', 'TicketGroupSize']

Missing values:
Fare               0
Age                0
Pclass             0
SibSp              0
Parch              0
Sex_male           0
Embarked_Q         0
Embarked_S         0
Survived           0
FamilySize         0
IsAlone            0
Title              0
TicketGroupSize    0
dtype: int64


## 13. Saving the Extracted Dataset

The result can now be used in the next Feature Engineering stage:

```text
Feature Selection
        ↓
Feature Extraction
        ↓
titanic_feature_extracted.csv
        ↓
Feature Transformation
```

In [ ]:
output_path = "../../Dataset/05_titanic_feature_extracted.csv"

df_extracted.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

Saved: ../../Dataset/titanic_feature_extracted.csv


## Key Takeaways

- Feature Extraction creates new features from existing information.
- `SibSp` + `Parch` → `FamilySize`
- `FamilySize` → `IsAlone`
- `Name` → `Title`
- `Ticket` → `TicketGroupSize`
- Raw columns can be removed after useful information has been extracted.

### Next

```text
Feature Selection
        ↓
Feature Extraction
        ↓
Feature Transformation
        ↓
Final ML Dataset
```